In [ ]:

from tqdm import tqdm
from model.classifier import Classifier
import os
import torch
import torch.nn.functional as F

from utils.transform_sequence import TransformSequence

device ="cuda" if torch.cuda.is_available() else "cpu"
def train(model, train_loader, optimizer):
    model.train()

    for data,y in tqdm(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data), data.y)
        loss.backward()
        optimizer.step()


def test(model, loader):
    model.eval()
    correct = 0
    for data,y in tqdm(loader):
        data = data.to(device)
        with torch.no_grad():
            pred = model(data).max(1)[1]
        correct += pred.eq(data.y).sum().item()
    return correct / len(loader.dataset)



In [ ]:
from torch_geometric.datasets import ModelNet
from torch_geometric.transforms import NormalizeScale
from torch_geometric.transforms import SamplePoints

pre_transform, transform = NormalizeScale(), SamplePoints(1024)
train_dataset_pre = ModelNet("../experiment_files/data/modelnet", '10', True, transform, pre_transform,force_reload=False)
val_dataset_pre = ModelNet("../experiment_files/data/modelnet", '10', False, transform, pre_transform,force_reload=False)

In [ ]:
from dataset.geometric_wrapper import GeometricLabelDatasetWrapper

train_dataset = GeometricLabelDatasetWrapper(train_dataset_pre)
val_dataset = GeometricLabelDatasetWrapper(val_dataset_pre)

In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,num_workers=0,persistent_workers=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,num_workers=0,persistent_workers=False)


In [ ]:
from torch_scatter import scatter

class BatchPointNormalizer(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, data):
        pos = data.pos
        batch_idx = data.batch

        # Center points
        mean = scatter(pos, batch_idx, dim=0, reduce="mean")
        pos = pos - mean[batch_idx]

        # Scale points
        dist = (pos ** 2).sum(dim=-1).sqrt()
        max_dist = scatter(dist, batch_idx, dim=0, reduce="max")
        pos = pos / (max_dist[batch_idx].unsqueeze(-1) + 1e-8)

        data.pos = pos
        return data


In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScale
from model.pointnet_plus import PointNetPlus

model_not_normalized = PointNetPlus()
model = torch.nn.Sequential(BatchNormalizeScale(),model_not_normalized)

In [ ]:
#train(model.to(device), train_loader, torch.optim.Adam(model.parameters(), lr=0.001))


In [ ]:
#pytorch set matmul precision
torch.set_float32_matmul_precision("medium")

In [ ]:
from model.classifier import MyProgressBar
import pytorch_lightning as pl
model_path = "../model/modelnet/modelnet10_normal_resampled.pth"

if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model_not_normalized.load_state_dict(torch.load(model_path))
else:
    print(f"Training model and will save to {model_path}")
    lightning_model = Classifier(model_not_normalized, optimizer_class =  torch.optim.AdamW, optimizer_params = {"lr": 1e-3})
    progress_bar = MyProgressBar()
    trainer = pl.Trainer(
        accelerator="cuda",
        max_epochs=50,
        precision="32",
        callbacks=[progress_bar],
    )
    # Train the model
    trainer.fit(lightning_model, train_loader,val_loader)
    # Test the model
    #trainer.test(lightning_model, test_loader)
    # Save model
    torch.save(model_not_normalized.state_dict(), model_path)
    print(f"Model saved to {model_path}")

In [ ]:
from model.pointnet_plus import SAModule
def set_deterministic_fps(model, random_start=False):
    for module in model.modules():
        if isinstance(module, SAModule):
            module.random_start = random_start

set_deterministic_fps(model)

In [ ]:
import plotly.graph_objects as go
def plot_interactive_3d(points):
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(size=2)
    )])
    fig.update_layout(scene=dict(aspectmode='data'))
    fig.show()

# Plot the 9th point cloud from the training dataset
plot_interactive_3d(train_dataset[8][0].pos.numpy())


In [ ]:
model.eval().to(device)
with torch.no_grad():
    test_acc = 0
    for data, target in val_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        test_acc += output.argmax(dim=-1).eq(data.y).sum().item()
    test_acc /= len(val_dataset)
    print(f'Mean accuracy on the val set: {test_acc}.')



In [ ]:
from utils.affine_transforms import AffineTransformations3D

transformations = [AffineTransformations3D.ROTATION.value,]#AffineTransformations3D.SCALING_ALL_SAME.value]
domains = [(-torch.pi,torch.pi)]

In [ ]:
from utils.transforms.apply import transform_3d_point_cloud

In [ ]:
samplex = next(iter(train_loader))[0].detach().cpu()
param = [torch.tensor([0.1,0.2,0.3],).unsqueeze(0), torch.tensor([0.8,]).unsqueeze(0),]
T = None
for i, transformation in enumerate(transformations):
            #get the transformation function
            transform_fun2 = transformation["matrix"]
            T = transform_fun2(T, param[i])

transformed = transform_3d_point_cloud(samplex[0].pos,T[0]).detach().cpu().numpy()

plot_interactive_3d(samplex[0].pos.detach().cpu().numpy())


In [ ]:
#plot both using interactive backend for jupyter
plot_interactive_3d(transformed)


In [ ]:
from utils.transform_sequence import TransformSequence
transformations = TransformSequence(transformations,domains=domains,application_method=transform_3d_point_cloud,device="cuda")

from utils.transformation_problem import create_sampler
transform_func = create_sampler(transformations)

In [ ]:
transform_func(10)

In [ ]:
transform_func(10)

In [ ]:


from dataset.mnist_no_pil import AffineTransformDataset
from dataset.geometric_wrapper import TensorGeometricsDatasetWrapper, TensorGeometricModelWrapper

wrapped_val = TensorGeometricsDatasetWrapper(val_dataset)


wrapped_model = TensorGeometricModelWrapper(model)

wrapped_val_affine_transformed = AffineTransformDataset(wrapped_val,transform_func,return_transformation=False,batch_size=32,resample_func=transform_3d_point_cloud)


wrapped_loader_val = DataLoader(wrapped_val, batch_size=32, shuffle=False)
wrapped_train = TensorGeometricsDatasetWrapper(train_dataset)
wrapped_loder_train = DataLoader(wrapped_train, batch_size=32, shuffle=True)




In [ ]:



from confidence.direct.logit_based import EnergyConfidence
from confidence.model.single_pass import SinglePassConfidence

confidence_module = SinglePassConfidence(wrapped_model, EnergyConfidence())
model.to(device)

In [ ]:
#test simulated annealing
from utils.transformation_problem import TransformationProblem


transformation_problem = TransformationProblem(confidence_module,transformations)

In [ ]:
transformations.device = "cuda"

In [ ]:
from search.simulated_anealing import ParallelSimulatedAnnealing

sim = ParallelSimulatedAnnealing(max_iterations=30,parallel_runs=30,initial_temp=0)
x,y= next(iter(wrapped_loader_val))

In [ ]:
with torch.no_grad():
    #get the transformed point cloud
    T = sim.optimize(transformation_problem,x[[2]].to(device),verbose=True)
    x_transfomred = transformation_problem.transform(x[[2]].to(device),T[0])
    cls_transformed = wrapped_model(x_transfomred).detach().cpu()
    cls_canon = wrapped_model(x[[2]].to(device)).detach().cpu()


In [ ]:
cls_transformed.detach().cpu()

In [ ]:
cls_canon.detach().cpu()

In [ ]:
energy_confidence = EnergyConfidence()

In [ ]:
energy_confidence(cls_transformed).detach().cpu()

In [ ]:
energy_confidence(cls_canon).detach().cpu()

In [ ]:
plot_interactive_3d(x_transfomred[0].cpu().numpy())

In [ ]:
#plot it
plot_interactive_3d(x[2].cpu().numpy())

In [ ]:
#trying another confidence measure

In [ ]:
list(wrapped_model.named_modules())

In [ ]:
from confidence.utils import ModelInputWrapper

#use model wrapper to extract the embeddings
dual_ouput_model = ModelInputWrapper(wrapped_model,'model.1.mlp.lins.1',flatten=True)

In [ ]:

import numpy as np
with torch.no_grad():
    embeddings=[]
    classes =[]
    for batch in tqdm(wrapped_loder_train):
        emb,_ = dual_ouput_model(batch[0].to(device))
        embeddings.append(emb.detach().cpu().numpy())
        classes.append(batch[1].detach().cpu().numpy())


    embeddings = np.vstack(embeddings)
    classes = np.hstack(classes)



In [ ]:
from confidence.unsupervised.classic.nn import NNDistanceConfidence

#test nearrest neigbhoor confidence
nearest_neighbor_confidence = NNDistanceConfidence(index_type="flat")
nearest_neighbor_confidence.fit(embeddings)
nearest_neighbor_confidence.to(device)

In [ ]:
from confidence.control.split import SplitConfidence

conf_split = SplitConfidence(nearest_neighbor_confidence,EnergyConfidence(), mult=True,b=1)
conf_2 = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem2 = TransformationProblem(conf_2,transformations)

# Create a new SimulatedAnnealing instance for the second confidence measure
sim2 = ParallelSimulatedAnnealing(max_iterations=30, parallel_runs=30, initial_temp=0)


In [ ]:
x[[2]]

In [ ]:
with torch.no_grad():
    T_true = transform_func(1)
    x2_sampled= transform_3d_point_cloud(x[[2]].to(device),T_true.to(device))

In [ ]:
x2_sampled

In [ ]:
with torch.no_grad():
    #get the transformed point cloud
    T = sim2.optimize(problem2,x2_sampled.to(device).clone(),verbose=True)
    x_transfomred = transformation_problem.transform(x2_sampled.to(device),T[0])
    cls = wrapped_model(x_transfomred).detach().cpu()
    cls_2 = wrapped_model(x2_sampled.to(device)).detach().cpu()


In [ ]:
cls.numpy()

In [ ]:
cls_2.numpy()

In [ ]:
plot_interactive_3d(x2_sampled[0].cpu().numpy())

In [ ]:
plot_interactive_3d(x_transfomred[0].cpu().numpy())

In [ ]:
#plot it
plot_interactive_3d(x[2].cpu().numpy())

In [ ]:
%%sql


In [ ]:
from search.parallel_gradient import WindowStuckDetectionDescent

pd = WindowStuckDetectionDescent(max_iterations=20, parallel_runs=10,learning_rate=0.1, lr_decay_rate = 1)

In [ ]:
from search.shgo import SHGO

In [ ]:
sh = SHGO(local_opt_kwargs={"lr":0.1},selection_method="delaunay")

In [ ]:

#get the transformed point cloud
T = pd.optimize(problem2,x2_sampled.to(device),verbose=True)
with torch.no_grad():
    x_transfomred = transformation_problem.transform(x2_sampled.to(device),T[0])
    cls = wrapped_model(x_transfomred).detach().cpu()
    cls_2 = wrapped_model(x2_sampled.to(device)).detach().cpu()


In [ ]:
x2_sampled

In [ ]:
cls.numpy()

In [ ]:
cls_2.numpy()

In [ ]:
plot_interactive_3d(x_transfomred[0].cpu().numpy())


In [ ]:
plot_interactive_3d(x[2].cpu().numpy())


In [ ]:
# from torch.utils.data import default_collate
#
# batch_size_opt = 2
# num_samples = len(wrapped_val)
# perm = torch.randperm(num_samples)
# transformed_dataset = []
#
# correct_total = 0
# samples_total = 0
# with torch.no_grad():
#     for i in range(0, 100, batch_size_opt):
#         # Extract a mini-batch using the permutation
#         batch_indices = perm[i : i + batch_size_opt].tolist()
#         batch = [wrapped_val[idx] for idx in batch_indices]
#         batch = default_collate(batch)
#
#         # Assuming the batch is a tuple (data, target)
#         x, target = batch
#         x = x.to(device)  # move data to GPU
#
#         T = sh.optimize(problem2, x)
#         x_transformed = problem2.transform(x, T)
#
#         _, logits = dual_ouput_model(x_transformed)
#         predictions = logits.argmax(dim=1)
#         correct = predictions.eq(target.to(device)).sum().item()
#         batch_acc = correct / target.size(0)
#
#         correct_total += correct
#         samples_total += target.size(0)
#
#     overall_accuracy = correct_total / samples_total
#     print(f'Overall accuracy on transformed dataset: {overall_accuracy:.4f}')


In [ ]:
# from torch.utils.data import default_collate
#
# batch_size_opt = 2
# num_samples = len(wrapped_val)
# perm = torch.randperm(num_samples)
# transformed_dataset = []
#
# correct_total = 0
# samples_total = 0
# with torch.no_grad():
#     for i in range(0, 100, batch_size_opt):
#         # Extract a mini-batch using the permutation
#         batch_indices = perm[i : i + batch_size_opt].tolist()
#         batch = [wrapped_val[idx] for idx in batch_indices]
#         batch = default_collate(batch)
#
#         # Assuming the batch is a tuple (data, target)
#         x, target = batch
#         x = x.to(device)  # move data to GPU
#
#         T = sim.optimize(problem2, x)
#         x_transformed = problem2.transform(x, T)
#
#         _, logits = dual_ouput_model(x_transformed)
#         predictions = logits.argmax(dim=1)
#         correct = predictions.eq(target.to(device)).sum().item()
#         batch_acc = correct / target.size(0)
#
#         correct_total += correct
#         samples_total += target.size(0)
#
#     overall_accuracy = correct_total / samples_total
#     print(f'Overall accuracy on transformed dataset: {overall_accuracy:.4f}')


In [ ]:
# from torch.utils.data import default_collate
#
# batch_size_opt = 2
# num_samples = len(wrapped_val)
# perm = torch.randperm(num_samples)
# transformed_dataset = []
#
# correct_total = 0
# samples_total = 0
# with torch.no_grad():
#     for i in range(0, 100, batch_size_opt):
#         # Extract a mini-batch using the permutation
#         batch_indices = perm[i : i + batch_size_opt].tolist()
#         batch = [wrapped_val[idx] for idx in batch_indices]
#         batch = default_collate(batch)
#
#         # Assuming the batch is a tuple (data, target)
#         x, target = batch
#         x = x.to(device)  # move data to GPU
#
#         T = pd.optimize(problem2, x)
#         x_transformed = problem2.transform(x, T)
#
#         _, logits = dual_ouput_model(x_transformed)
#         predictions = logits.argmax(dim=1)
#         correct = predictions.eq(target.to(device)).sum().item()
#         batch_acc = correct / target.size(0)
#
#         correct_total += correct
#         samples_total += target.size(0)
#
#     overall_accuracy = correct_total / samples_total
#     print(f'Overall accuracy on transformed dataset: {overall_accuracy:.4f}')


In [ ]:
torch.cuda.empty_cache()

In [ ]:
from confidence.input_transform import InputTransform
ip = InputTransform(standardize=True)
ip.fit(embeddings)

In [ ]:
ip.cuda()

In [ ]:
nearest_neighbor_confidence2 = NNDistanceConfidence(index_type="flat",number_of_neighbors=5,input_transform=ip)
nearest_neighbor_confidence2.fit(embeddings)
nearest_neighbor_confidence2.to(device)

In [ ]:
conf_split_2 = SplitConfidence(nearest_neighbor_confidence2,EnergyConfidence(), mult=False,a=0.0)
conf_only_conf = SinglePassConfidence(dual_ouput_model,conf_split_2,index=1)

In [ ]:
conf_split_2 = SplitConfidence(nearest_neighbor_confidence2,EnergyConfidence(), mult=False,b=0.0)
conf_only_nn = SinglePassConfidence(dual_ouput_model,conf_split_2,index=1)

In [ ]:
conf_split_2 = SplitConfidence(nearest_neighbor_confidence2,EnergyConfidence(), mult=True)
conf_both = SinglePassConfidence(dual_ouput_model,conf_split_2,index=1)

In [ ]:
#note found that gmm and nn is not robust against outliers in data. May need a way to filter them one class svm could be reused for this(or isolation forest as it has auto finding)
#note that osvm requires there to be outlier to function(maybe artifically add wrong ones?), iso handles outliers as well.

In [ ]:
def calculate_confidences(problem2, conf_fn, x, transformations, resolution=21, batch_size=1, device="cuda"):
    """
    Computes the confidences over a grid of transformation parameters.
    Args:
        problem2: Transformation problem with an application_method.
        conf_fn: A confidence function e.g. conf_only_conf.
        x: Input tensor (batch) on which to apply the transformation.
        transformations: List of transformation dictionaries, each with a "matrix" function.
        resolution: Number of grid partitions along an axis.
        batch_size: Batch size to compute confidences.
        device: Device to use.
    Returns:
        confidences: flattened confidences for the grid.
        params: flattened parameters used for the grid.
    """
    # create grid parameters between -pi and pi
    lin = np.linspace(-np.pi, np.pi, resolution)
    x_lin, y_lin, z_lin = lin, lin, lin
    x_mesh, y_mesh, z_mesh = np.meshgrid(x_lin, y_lin, z_lin)
    x_flat = x_mesh.flatten()
    y_flat = y_mesh.flatten()
    z_flat = z_mesh.flatten()

    # generate parameter vector and convert to tensor
    p = np.vstack([x_flat, y_flat, z_flat]).T
    p = torch.tensor(p, dtype=torch.float32, device=device)

    # calculate transformation matrix
    T = None
    for transformation in transformations.transformations:
        transform_fun = transformation["matrix"]
        T = transform_fun(T, p)  # expected shape: (N, 4, 4)

    # for each transformation matrix, compute confidence in batches
    total = T.shape[0]
    result_list = []
    with torch.no_grad():
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            T_batch = T[start:end]
            # Apply transformation using application_method.
            # Here, x should have shape compatible with problem2.application_method.
            x_transformed = problem2.transform_sequence.application_method(x, T_batch)
            # conf_fn returns (confidences, logits). We average over the sample dimensions.
            conf, _ = conf_fn(x_transformed)
            result_list.append(conf.detach().cpu().numpy())


    confidences_flat = np.concatenate(result_list, axis=0)
    confidences = confidences_flat
    return confidences, p.detach().cpu().numpy()


def draw_volume(params, confidences, use_log_scale=False, eps=1e-12):
    """
    Plots the 3D volume using Plotly Volume.

    Args:
        params:    NumPy array of shape (N, 3) with x, y, z coordinates.
        confidences: 3D NumPy array of shape (resolution, resolution, resolution).
        use_log_scale: If True, plot log10(confidences + eps) instead of raw.
        eps:       Small offset to add before taking log to avoid log(0).
    """
    # flatten the grid
    x = params[:, 0]
    y = params[:, 1]
    z = params[:, 2]
    flat_conf = confidences.flatten()

    values = flat_conf
    if use_log_scale:
        #do log modulo tranform
        values = np.sign(values) * np.log(1 + np.abs(values))


    vmin, vmax = np.min(values), np.max(values)
    cbar_title = 'Confidence'

    fig = go.Figure(
        data=go.Volume(
            x=x,
            y=y,
            z=z,
            value=values,
            isomin=vmin,
            isomax=vmax,
            opacity=0.1,
            surface_count=20,
            colorscale='Viridis',
            colorbar=dict(title=cbar_title)
        )
    )

    fig.update_layout(
        scene=dict(
            xaxis_title='Rotation X',
            yaxis_title='Rotation Y',
            zaxis_title='Rotation Z',
            aspectmode='manual'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        width=800,
        height=800
    )
    fig.show()

In [ ]:


from confidence.unsupervised.classic.gmm import GaussianMixtureConfidence

gmm_confidence = GaussianMixtureConfidence(n_components=500,covariance_type='spherical',input_transform=ip)
gmm_confidence.fit(embeddings)
gmm_confidence.cuda()

conf_split_2 = SplitConfidence(gmm_confidence, EnergyConfidence(), mult=False, b=0.0)
conf_only_gmm = SinglePassConfidence(dual_ouput_model, conf_split_2, index=1)
conf_split_2 = SplitConfidence(gmm_confidence, EnergyConfidence(), mult=True)
conf_gmm_both = SinglePassConfidence(dual_ouput_model, conf_split_2, index=1)

In [ ]:
def normalize_with_skew(arr, strength=0.0):
    arr_min = np.min(arr)
    arr_max = np.max(arr)

    if arr_max == arr_min:
        return np.zeros_like(arr)  # Avoid division by zero for constant arrays

    normalized = (arr - arr_min) / (arr_max - arr_min)

    if strength > 0:
        skew_factor = 1 + strength
        normalized = normalized ** skew_factor

    return normalized


In [ ]:
x_rand,label = wrapped_train[torch.randint(0, len(wrapped_train), (1,)).item()]

In [ ]:
x_rand = x_rand.unsqueeze(0).to(device)

In [ ]:
cs = []
with torch.no_grad():
    for i in range(200):
        par = transformation_problem.initial_param(1).cuda()
        c= conf_only_gmm(problem2.transform(x_rand, par).cuda())[0]
        cs.append(c)




In [ ]:
print(torch.max(torch.cat(cs)).item())

In [ ]:
conf_only_gmm(x_rand)

In [ ]:
plot_interactive_3d(x_rand[0].cpu().numpy())


In [ ]:
conf, p = calculate_confidences(problem2, conf_only_conf.cuda(), x_rand, transformations, resolution=21, batch_size=32, device="cuda")
draw_volume(p, conf)

In [ ]:
#draw the volume


In [ ]:
conf, p = calculate_confidences(problem2, conf_only_nn, x_rand, transformations, resolution=21, batch_size=32,
                                device="cuda")
#draw the volume
draw_volume(p, conf)

In [ ]:
conf, p = calculate_confidences(problem2, conf_both, x_rand, transformations, resolution=21, batch_size=32,
                                device="cuda")


In [ ]:
#draw the volume
draw_volume(p, conf)